In [0]:

# Chennai Urban Intelligence
# Silver Data Quality Validation

silver_path = "/Volumes/workspace/default/chennai_silver_input/"

datasets = {
    "rainfall": spark.read.option("header", True).option("inferSchema", True).csv(silver_path + "rainfall_cleaned.csv"),
    "flood": spark.read.option("header", True).option("inferSchema", True).csv(silver_path + "flood_cleaned.csv"),
    "road_complaints": spark.read.option("header", True).option("inferSchema", True).csv(silver_path + "road_complaints_cleaned.csv"),
    "traffic": spark.read.option("header", True).option("inferSchema", True).csv(silver_path + "traffic_cleaned.csv"),
    "public_grievances": spark.read.option("header", True).option("inferSchema", True).csv(silver_path + "public_grievances_cleaned.csv"),
    "waterlogging": spark.read.option("header", True).option("inferSchema", True).csv(silver_path + "waterlogging_locations_cleaned.csv"),
    "population": spark.read.option("header", True).option("inferSchema", True).csv(silver_path + "population_cleaned.csv"),
    "emergency_response": spark.read.option("header", True).option("inferSchema", True).csv(silver_path + "emergency_response_cleaned.csv")
}

print("All datasets loaded successfully")


In [0]:

from pyspark.sql.functions import col, sum, when

for name, df in datasets.items():

    print("\nDataset:", name)

    null_counts = df.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])

    display(null_counts)


In [0]:


for name, df in datasets.items():

    total_records = df.count()

    unique_records = df.dropDuplicates().count()

    duplicate_records = total_records - unique_records

    print("----------------------------------")
    print("Dataset:", name)
    print("Total Records:", total_records)
    print("Unique Records:", unique_records)
    print("Duplicate Records:", duplicate_records)


In [0]:

for name, df in datasets.items():

    print("\n================================")
    print("DATASET:", name)
    print("================================")

    df.printSchema()


In [0]:

# Save validated datasets as Delta tables

for name, df in datasets.items():

    table_name = "silver_" + name

    df.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable("workspace.default." + table_name)

    print("Saved:", table_name)


In [0]:

# Verify saved Delta tables

for name in datasets.keys():

    table_name = "workspace.default.silver_" + name

    df = spark.table(table_name)

    print(table_name, ":", df.count(), "records")
